# Stage 1: Select Implied Copyable Trades

Find profitable follower BUYs that follow a leader's BUY or SELL on the same
token within a time window. Two separate leader groups: **buy leaders** (whose
BUYs precede follower BUYs) and **sell leaders** (whose SELLs precede follower
BUYs).

Grid-search over selection thresholds to maximize **copyable PnL from implied
trades** on the validation split.

**Output:** `stage1_implied_result.json` with best selection params.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    select_follower_wallets,
    select_leader_wallets,
    detect_implied_buys,
    score_leaders,
    evaluate_implied_pnl,
    evaluate_follower_buy_performance,
    iterative_leader_follower_filter,
    filter_stable_leaders,
    filter_leaders_by_drawdown,
    filter_pairs_by_frequency,
    run_implied_grid_search,
    DEFAULT_TAGS
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

## Parameters

In [2]:
if DEFAULT_TAGS == {"Weather"}:
    PARAMS = dict(
        # --- Follower selection ---
        min_follower_copyable_roi=0.30,
        min_follower_trade_value=100,
        min_follower_num_buckets=30,
        max_follower_hhi=0.3,
        # --- Leader selection ---
        min_leader_trade_count=20,
        max_leader_hhi=1,
        # --- Broad follower set for leader stability detection ---
        stability_min_follower_roi=0.0,
        stability_min_follower_buckets=10,
        stability_max_follower_hhi=1,
        # --- Leader filtering (drawdown-based) ---
        max_dd_pnl_ratio=0.3,
        # --- Pair filtering ---
        min_pair_observations=3,
        # --- Time window ---
        time_window_minutes=15,
    )
elif DEFAULT_TAGS == {"Politics"}:
    PARAMS = dict(
        # --- Follower selection ---
        min_follower_copyable_roi=0.30,
        min_follower_trade_value=100,
        min_follower_num_buckets=30,
        max_follower_hhi=0.3,
        # --- Leader selection ---
        min_leader_trade_count=20,
        max_leader_hhi=1,
        # --- Broad follower set for leader stability detection ---
        stability_min_follower_roi=0.0,
        stability_min_follower_buckets=10,
        stability_max_follower_hhi=1,
        # --- Stability filtering ---
        stability_n_splits=3,
        stability_min_profitable_splits=2,
        # --- Pair filtering ---
        min_pair_observations=3,
        # --- Time window ---
        time_window_minutes=30,
    )
else:
    raise ValueError(f"Unsupported DEFAULT_TAGS: {DEFAULT_TAGS}")

for k, v in PARAMS.items():
    print(f"  {k}: {v}")

  min_follower_copyable_roi: 0.3
  min_follower_trade_value: 100
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_leader_trade_count: 20
  max_leader_hhi: 1
  stability_min_follower_roi: 0.0
  stability_min_follower_buckets: 10
  stability_max_follower_hhi: 1
  max_dd_pnl_ratio: 0.3
  min_pair_observations: 3
  time_window_minutes: 15


## Load data

In [3]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)

train_cutoff = pd.Timestamp("2026-06-01", tz="UTC")
val_cutoff = pd.Timestamp("2026-07-01", tz="UTC")

df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Split by trade date:")
print(f"  Train: {len(df_train):>10,} trades  ({df_train['condition_id'].nunique():>5,} markets)  < {train_cutoff.date()}")
print(f"  Val:   {len(df_val):>10,} trades  ({df_val['condition_id'].nunique():>5,} markets)  {train_cutoff.date()} .. {val_cutoff.date()}")
print(f"  Test:  {len(df_test):>10,} trades  ({df_test['condition_id'].nunique():>5,} markets)  >= {val_cutoff.date()}")
print(f"  Total: {len(df_full):>10,} trades  ({df_full['condition_id'].nunique():>5,} markets)")

# Market overlap check
train_markets = set(df_train["condition_id"].unique())
val_markets = set(df_val["condition_id"].unique())
test_markets = set(df_test["condition_id"].unique())
print(f"\n  Markets overlapping train/val: {len(train_markets & val_markets)}")
print(f"  Markets overlapping train/test: {len(train_markets & test_markets)}")
print(f"  Markets overlapping val/test: {len(val_markets & test_markets)}")

Markets: 1877548


Filtered markets for {'Weather'}: 87967
Loading 16 trade shards...


Total trades loaded: 13,603,198


Unique wallets: 4,054
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-22 05:47:10+00:00


Split by trade date:


  Train:  6,035,492 trades  (23,237 markets)  < 2026-06-01
  Val:    4,766,255 trades  (19,787 markets)  2026-06-01 .. 2026-07-01


  Test:   2,801,451 trades  (13,190 markets)  >= 2026-07-01


  Total: 13,603,198 trades  (53,732 markets)



  Markets overlapping train/val: 1350
  Markets overlapping train/test: 1
  Markets overlapping val/test: 1132


## Compute wallet metrics on training data

In [4]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3584


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.0236,NaN,13.2971,0.0193,45
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0051,0.0020,-3.2519,0.0000,212
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0262,0.0360,229.8169,-0.0020,5300
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,NaN,0.0000,NaN,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.6776,-1.0000,115.4494,0.4865,34
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,NaN,-20.9264,NaN,299
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0120,-1.0000,17.6344,-0.0668,46
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0259,-0.1815,110.7907,0.0051,4360
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.1968,-0.0635,148.3132,0.0086,1356
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.0058,-0.9697,25.1798,0.0182,2454


## Baseline selection

In [5]:
follower_wallets = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=PARAMS["min_follower_copyable_roi"],
    min_trade_value=PARAMS["min_follower_trade_value"],
    min_num_buckets=PARAMS["min_follower_num_buckets"],
    max_market_pnl_hhi=PARAMS["max_follower_hhi"],
)
print(f"Followers: {len(follower_wallets)}")

buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=PARAMS["min_leader_trade_count"],
    min_roi=None,
    max_market_pnl_hhi=PARAMS["max_leader_hhi"],
    side="BUY",
)
print(f"Buy leaders: {len(buy_leaders)}")

sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=PARAMS["min_leader_trade_count"],
    min_roi=None,
    max_market_pnl_hhi=PARAMS["max_leader_hhi"],
    side="SELL",
)
print(f"Sell leaders: {len(sell_leaders)}")

Followers: 44
Buy leaders: 1401
Sell leaders: 1401


## Baseline evaluation

In [6]:
follower_ws = set(follower_wallets['wallet'])
buy_leader_ws = set(buy_leaders['wallet'])
sell_leader_ws = set(sell_leaders['wallet'])
tw = PARAMS["time_window_minutes"]

for split_name, df_split in  [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(
        df_split, follower_ws, buy_leader_ws,
        time_window_minutes=tw, leader_side="BUY"
    )
    sell_ev = evaluate_implied_pnl(
        df_split, follower_ws, sell_leader_ws,
        time_window_minutes=tw, leader_side="SELL",
    )
    total = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    roi = total / notional if notional > 0 else 0
    print(f"{split_name}: buy_pnl={buy_ev['followed_copyable_pnl']:.2f} ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)  "
          f"sell_pnl={sell_ev['followed_copyable_pnl']:.2f} ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)  "
          f"total={total:.2f}  roi={roi:.4f}")

TRAIN: buy_pnl=33457.17 (10816 trades, 735 leaders)  sell_pnl=35405.32 (13835 trades, 545 leaders)  total=68862.49  roi=0.5427


VAL: buy_pnl=3420.21 (9251 trades, 525 leaders)  sell_pnl=4409.03 (11566 trades, 412 leaders)  total=7829.23  roi=0.0423


TEST: buy_pnl=7909.45 (6359 trades, 397 leaders)  sell_pnl=10242.99 (8178 trades, 277 leaders)  total=18152.44  roi=0.1920


## Score leaders (baseline)

In [7]:
# Buy leaders
buy_implied = detect_implied_buys(
    df_train, follower_ws, buy_leader_ws,
    time_window_minutes=tw, leader_side="BUY",
)
buy_scores = score_leaders(buy_implied)
print("Top buy leaders:")
print(buy_scores.head(10).to_string())

print()

# Sell leaders
sell_implied = detect_implied_buys(
    df_train, follower_ws, sell_leader_ws,
    time_window_minutes=tw, leader_side="SELL",
)
sell_scores = score_leaders(sell_implied)
print("Top sell leaders:")
print(sell_scores.head(10).to_string())

Top buy leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0x3dc44175ae2d0175ee7bc76cd9ec04b614a91fd8              9                    4089.5131                   30             22
1  0xde0fa43faad1c0da4881c63681ad07a8c850a4e7              1                    2227.6980                    1              1
2  0x48180cfe026031c280ebdea2acad996867cde5de             15                    1961.8385                   85             53
3  0x36901eb0f21519cc9055662a6d2483e96da1e16f             16                    1909.9095                   41             36
4  0x57ee70867b4e387de9de34fd62bc685aa02a8112              7                    1825.7647                   17             15
5  0x50b977391c4b3dd88b0a0bef03c3434fe4284298             13                    1537.4340                   46             40
6  0xa3282d3e882501229c75d0caf134e62e3afb4977              2                    1525.0000            

Top sell leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0x177500541ae20bb0d46ab0db3fd2559e2a7e85b0              3                    9515.1000                   11              3
1  0xb40e89677d59665d5188541ad860450a6e2a7cc9             41                    6465.6144                 1252            525
2  0x945a49252f772a10c6ddd1d1e1e24ee20438a48c             39                    3214.9853                 1170            488
3  0x48180cfe026031c280ebdea2acad996867cde5de             16                    2541.3673                  268             59
4  0xc34f6b088bb9172625ee1ea2ee8da9ac4f037d2e             32                    2504.5069                  391            181
5  0x26123cbf0f4820f7e70408a8c054ba7615c05289             37                    2011.5062                 1753            486
6  0x77bdeb3f229bf6d826d20dbd5f0c7972c32ae48f             25                    1712.6761           

## Improved pipeline: Stable leaders + pair filtering

Filter leaders who are consistently profitable across training time slices,
then require a minimum number of observed copy-trades per leader-follower pair.

In [8]:
# Broad follower set for leader stability detection (more implied trades per leader)
fw_broad = set(select_follower_wallets(
    wallet_vol,
    min_copyable_roi=PARAMS["stability_min_follower_roi"],
    min_trade_value=PARAMS["min_follower_trade_value"],
    min_num_buckets=PARAMS["stability_min_follower_buckets"],
    max_market_pnl_hhi=PARAMS["stability_max_follower_hhi"],
)["wallet"])

if DEFAULT_TAGS == {"Weather"}:
    stable_buy_leaders = filter_leaders_by_drawdown(
        df_train, buy_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="BUY",
        max_dd_pnl_ratio=PARAMS["max_dd_pnl_ratio"],
    )
    stable_sell_leaders = filter_leaders_by_drawdown(
        df_train, sell_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="SELL",
        max_dd_pnl_ratio=PARAMS["max_dd_pnl_ratio"],
    )
elif DEFAULT_TAGS == {"Politics"}:
    stable_buy_leaders = filter_stable_leaders(
        df_train, buy_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="BUY",
        n_splits=PARAMS["stability_n_splits"],
        min_profitable_splits=PARAMS["stability_min_profitable_splits"],
    )
    stable_sell_leaders = filter_stable_leaders(
        df_train, sell_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="SELL",
        n_splits=PARAMS["stability_n_splits"],
        min_profitable_splits=PARAMS["stability_min_profitable_splits"],
    )
else:
    raise ValueError(f"Unsupported DEFAULT_TAGS: {DEFAULT_TAGS}")

print(f"Stable buy leaders: {len(stable_buy_leaders)} / {len(buy_leader_ws)}")
print(f"Stable sell leaders: {len(stable_sell_leaders)} / {len(sell_leader_ws)}")

buy_imp_train = detect_implied_buys(
    df_train, follower_ws, stable_buy_leaders,
    time_window_minutes=tw, leader_side="BUY",
)
sell_imp_train = detect_implied_buys(
    df_train, follower_ws, stable_sell_leaders,
    time_window_minutes=tw, leader_side="SELL",
)

min_obs = PARAMS["min_pair_observations"]
buy_pairs = filter_pairs_by_frequency(buy_imp_train, min_observations=min_obs)
sell_pairs = filter_pairs_by_frequency(sell_imp_train, min_observations=min_obs)

final_buy_leaders = set(buy_pairs["leader_wallet"]) if not buy_pairs.empty else set()
final_sell_leaders = set(sell_pairs["leader_wallet"]) if not sell_pairs.empty else set()
final_followers = (
    (set(buy_pairs["follower_wallet"]) if not buy_pairs.empty else set())
    | (set(sell_pairs["follower_wallet"]) if not sell_pairs.empty else set())
)

print()  # newline before summary
print(f"After pair filter (min_obs={min_obs}):")
print(f"  Followers: {len(final_followers)}")
print(f"  Buy leaders: {len(final_buy_leaders)}")
print(f"  Sell leaders: {len(final_sell_leaders)}")

Stable buy leaders: 550 / 1401
Stable sell leaders: 422 / 1401



After pair filter (min_obs=3):
  Followers: 43
  Buy leaders: 132
  Sell leaders: 73


In [9]:
print("=" * 70)
print(f"EVALUATION (tw={tw}min, min_pair_obs={min_obs})")
print("=" * 70)

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(df_split, final_followers, final_buy_leaders, time_window_minutes=tw, leader_side="BUY")
    sell_ev = evaluate_implied_pnl(df_split, final_followers, final_sell_leaders, time_window_minutes=tw, leader_side="SELL")
    follower_buy = evaluate_follower_buy_performance(df_split, final_followers)

    n_active = len(set(df_split[df_split["wallet"].isin(final_followers)]["wallet"]))
    n_markets = df_split["condition_id"].nunique()

    imp_pnl = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    imp_notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    imp_trades = buy_ev["trade_count"] + sell_ev["trade_count"]
    imp_roi = imp_pnl / imp_notional if imp_notional > 0 else 0.0

    b_roi = buy_ev["followed_copyable_pnl"] / buy_ev["followed_copyable_notional"] if buy_ev["followed_copyable_notional"] > 0 else 0.0
    s_roi = sell_ev["followed_copyable_pnl"] / sell_ev["followed_copyable_notional"] if sell_ev["followed_copyable_notional"] > 0 else 0.0

    print(f"\n{split_name} ({n_markets} markets, {n_active} active followers):")
    print(f"  Implied BUY:   Copyable PnL: {buy_ev['followed_copyable_pnl']:>10.2f}  ROI: {b_roi:>7.4f}  ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)")
    print(f"  Implied SELL:  Copyable PnL: {sell_ev['followed_copyable_pnl']:>10.2f}  ROI: {s_roi:>7.4f}  ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)")
    print(f"  Implied total: Copyable PnL: {imp_pnl:>10.2f}  ROI: {imp_roi:>7.4f}  ({imp_trades} trades)")
    print(f"  All buys:      Copyable PnL: {follower_buy['followed_copyable_pnl']:>10.2f}  ROI: {follower_buy['followed_copyable_roi']:>7.4f}  (wallet PnL: {follower_buy['wallet_pnl']:>10.2f}, {follower_buy['trade_count']} trades)")

EVALUATION (tw=15min, min_pair_obs=3)



TRAIN (23237 markets, 43 active followers):
  Implied BUY:   Copyable PnL:   32623.47  ROI:  0.5867  (12153 trades, 132 leaders)
  Implied SELL:  Copyable PnL:   36228.27  ROI:  0.5820  (11868 trades, 73 leaders)
  Implied total: Copyable PnL:   68851.74  ROI:  0.5842  (24021 trades)
  All buys:      Copyable PnL:   39700.41  ROI:  0.4926  (wallet PnL:   86340.40, 28757 trades)



VAL (19787 markets, 28 active followers):
  Implied BUY:   Copyable PnL:    2361.88  ROI:  0.0339  (7454 trades, 95 leaders)
  Implied SELL:  Copyable PnL:    3632.08  ROI:  0.0439  (9805 trades, 53 leaders)
  Implied total: Copyable PnL:    5993.96  ROI:  0.0394  (17259 trades)
  All buys:      Copyable PnL:    4610.68  ROI:  0.0369  (wallet PnL:   40916.44, 20953 trades)



TEST (13190 markets, 26 active followers):
  Implied BUY:   Copyable PnL:    8378.60  ROI:  0.2293  (4938 trades, 77 leaders)
  Implied SELL:  Copyable PnL:   10403.02  ROI:  0.2228  (7209 trades, 44 leaders)
  Implied total: Copyable PnL:   18781.62  ROI:  0.2256  (12147 trades)
  All buys:      Copyable PnL:   11211.19  ROI:  0.1619  (wallet PnL:   49462.96, 15611 trades)


## Summary

In [10]:
# Store for save cell
b_fw = final_followers
b_blw = final_buy_leaders
b_slw = final_sell_leaders

print(f"Final wallet counts:")
print(f"  Followers: {len(b_fw)}")
print(f"  Buy leaders: {len(b_blw)}")
print(f"  Sell leaders: {len(b_slw)}")

Final wallet counts:
  Followers: 43
  Buy leaders: 132
  Sell leaders: 73


In [11]:
best_params = PARAMS.copy()
print("Pipeline params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Pipeline params:
  min_follower_copyable_roi: 0.3
  min_follower_trade_value: 100
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_leader_trade_count: 20
  max_leader_hhi: 1
  stability_min_follower_roi: 0.0
  stability_min_follower_buckets: 10
  stability_max_follower_hhi: 1
  max_dd_pnl_ratio: 0.3
  min_pair_observations: 3
  time_window_minutes: 15


## Concentration diagnostics (test split)

In [12]:
buy_imp_test = detect_implied_buys(df_test, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
sell_imp_test = detect_implied_buys(df_test, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")
imp_test = pd.concat([buy_imp_test, sell_imp_test], ignore_index=True)

if imp_test.empty:
    print("No implied trades on test set.")
else:
    total_pnl = imp_test["copyable_pnl"].sum()
    total_trades = len(imp_test)
    print(f"Test implied trades: {total_trades:,}   Total copyable PnL: ${total_pnl:,.2f}\n")

    # --- Leader concentration ---
    leader_pnl = (
        imp_test.groupby("leader_wallet", sort=False)["copyable_pnl"]
        .agg(["sum", "count", "nunique"])
        .rename(columns={"sum": "pnl", "count": "trades", "nunique": "followers"})
        .sort_values("pnl", ascending=False)
    )
    leader_pnl["cum_pnl"] = leader_pnl["pnl"].cumsum()
    leader_pnl["cum_pct"] = leader_pnl["cum_pnl"] / total_pnl
    n_leaders = len(leader_pnl)
    print(f"Leaders contributing to test PnL: {n_leaders}")
    for k in [1, 3, 5, 10]:
        if k <= n_leaders:
            pct = leader_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} leader(s): ${leader_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 leaders:")
    print(leader_pnl.head(10).to_string())

    # --- Follower concentration ---
    follower_pnl = (
        imp_test.groupby("follower_wallet", sort=False)["copyable_pnl"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pnl", "count": "trades"})
        .sort_values("pnl", ascending=False)
    )
    follower_pnl["cum_pnl"] = follower_pnl["pnl"].cumsum()
    follower_pnl["cum_pct"] = follower_pnl["cum_pnl"] / total_pnl
    n_followers = len(follower_pnl)
    print(f"\nFollowers active on test: {n_followers}")
    for k in [1, 5, 10, 20]:
        if k <= n_followers:
            pct = follower_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} follower(s): ${follower_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 followers:")
    print(follower_pnl.head(10).to_string())

    # --- Market concentration ---
    market_pnl = (
        imp_test.groupby("condition_id", sort=False)["copyable_pnl"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pnl", "count": "trades"})
        .sort_values("pnl", ascending=False)
    )
    market_pnl["cum_pnl"] = market_pnl["pnl"].cumsum()
    market_pnl["cum_pct"] = market_pnl["cum_pnl"] / total_pnl
    n_markets = len(market_pnl)
    print(f"\nMarkets with implied trades: {n_markets}")
    for k in [1, 3, 5, 10]:
        if k <= n_markets:
            pct = market_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} market(s):  ${market_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 markets:")
    print(market_pnl.head(10).to_string())

    # --- Negative PnL followers ---
    neg_followers = (follower_pnl["pnl"] < 0).sum()
    neg_pnl = follower_pnl.loc[follower_pnl["pnl"] < 0, "pnl"].sum()
    print(f"\nFollowers with negative PnL: {neg_followers}  (total: ${neg_pnl:,.2f})")
    pos_followers = (follower_pnl["pnl"] > 0).sum()
    pos_pnl = follower_pnl.loc[follower_pnl["pnl"] > 0, "pnl"].sum()
    print(f"Followers with positive PnL: {pos_followers}  (total: ${pos_pnl:,.2f})")

    # --- Gini coefficient on leader PnL ---
    vals = leader_pnl["pnl"].values
    vals_sorted = np.sort(vals)
    n = len(vals_sorted)
    cum = np.cumsum(vals_sorted)
    gini = 1 - 2 * np.sum(cum) / (n * cum[-1]) if cum[-1] > 0 else 0.0
    print(f"\nLeader PnL Gini coefficient: {gini:.4f}  (1 = perfect concentration, 0 = equal)")

Test implied trades: 12,147   Total copyable PnL: $18,781.62

Leaders contributing to test PnL: 105
  Top  1 leader(s): $  8,631.63  (46.0% of total)
  Top  3 leader(s): $ 12,982.50  (69.1% of total)
  Top  5 leader(s): $ 15,428.03  (82.1% of total)
  Top 10 leader(s): $ 17,445.27  (92.9% of total)

Top 10 leaders:
                                                 pnl  trades  followers    cum_pnl  cum_pct
leader_wallet                                                                              
0x945a49252f772a10c6ddd1d1e1e24ee20438a48c 8631.6333    3702       1924  8631.6333   0.4596
0xb40e89677d59665d5188541ad860450a6e2a7cc9 2345.7092    1806       1067 10977.3425   0.5845
0x26123cbf0f4820f7e70408a8c054ba7615c05289 2005.1568    1837        851 12982.4992   0.6912
0x510f4963b66b1b18505faab74b0bb943d1dda43c 1678.3531     574        290 14660.8523   0.7806
0xfb5c08039ac3fde6bcc946bc1ac9823ab80ecda7  767.1731     140        126 15428.0255   0.8214
0xcd154f053a291f46e550d5fbcd4cfa7ddbc4b

## Save stage 1 result

In [13]:
import json
from datetime import datetime, timezone
from pathlib import Path


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# Collect wallet records for each group
wallet_cols = [
    "wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "market_pnl_hhi",
]

def _wallet_records(df):
    if df is None or df.empty:
        return []
    cols = [c for c in wallet_cols if c in df.columns]
    records = df[cols].to_dict(orient="records")
    return [{k: _convert(v) for k, v in w.items()} for w in records]


metadata = {
    "type": "implied",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_followers": len(b_fw),
    "n_buy_leaders": len(b_blw),
    "n_sell_leaders": len(b_slw),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "type": "implied",
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "metadata": metadata,
    "wallets": {
        "followers": _wallet_records(follower_wallets[follower_wallets["wallet"].isin(b_fw)]),
        "buy_leaders": _wallet_records(buy_leaders[buy_leaders["wallet"].isin(b_blw)]),
        "sell_leaders": _wallet_records(sell_leaders[sell_leaders["wallet"].isin(b_slw)]),
    },
}

out_path = Path("stage1_implied_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")

Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_implied_result.json
